<a href="https://colab.research.google.com/github/rybak97/free_courses/blob/main/Q_learning_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

based on: https://www.datacamp.com/tutorial/introduction-q-learning-beginner-tutorial

# What is Q-Learning

In [ ]:
import os
from google.colab import drive

# Mount Google Drive to save the HTML file
drive.mount('/content/drive')

# Define the input path for the current notebook and the output path for the HTML file
# This assumes your notebook is in 'My Drive/Colab Notebooks/'. Adjust if needed.
current_notebook_name = 'Q-learning Tutorial.ipynb' # Ensure this matches your notebook's name
input_notebook_path = f'/content/drive/MyDrive/Colab Notebooks/{current_notebook_name}'
output_html_path = '/content/drive/MyDrive/Q-learning_Tutorial.html'

print(f"Input notebook path: {input_notebook_path}")
print(f"Output HTML path: {output_html_path}")

Q-learning is a **model-free, value-based, off-policy** algorithm that will find the best series of actions based on the agent's current state.

The model-based algorithms use transition and reward functions to estimate the optimal policy and create the model. In contrast, model-free algorithms learn the consequences of their actions through the experience without transition and reward function.

The value-based method trains the value function to learn which state is more valuable and take action. On the other hand, policy-based methods train the policy directly to learn which action to take in a given state

In the off-policy, the algorithm evaluates and updates a policy that differs from the policy used to take an action. Conversely, the on-policy algorithm evaluates and improves the same policy used to take an action.  

# Example w/ Pygame

In [ ]:
!pip install pyglet==1.5.1
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
!pip install gym==0.24
!pip install pygame
!pip install numpy

!pip install imageio imageio_ffmpeg

In [ ]:
import numpy as np
import gym
import random
import imageio
from tqdm.notebook import trange

In [ ]:
env = gym.make("FrozenLake-v1",map_name="4x4",is_slippery=False)

print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample()) # display a random observation

In [ ]:
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample())

In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

action_space = env.action_space.n
print("There are ", action_space, " possible actions")

In [ ]:
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space, action_space))
  return Qtable

Qtable_frozenlake = initialize_q_table(state_space, action_space)

In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  random_int = random.uniform(0,1)
  if random_int > epsilon:
    action = np.argmax(Qtable[state])
  else:
    action = env.action_space.sample()
  return action

In [ ]:
def greedy_policy(Qtable, state):
  action = np.argmax(Qtable[state])
  return action

## Model hyperparameters

There are 10,000 training and 100 evaluation episodes.

The learning rate is 0.7.

We are using "FrozenLake-v1" as an environment with 99 maximum steps per episode.

The gamma (discount rate) is 0.95.

eval_seed: evaluation seed for the environment.

The exploration epsilon probability at the start is 1.0, and the minimum probability will be 0.05.

The exponential decay rate for epsilon probability is 0.0005.

In [ ]:
# Training parameters
n_training_episodes = 10000
learning_rate = 0.7

# Evaluation parameters
n_eval_episodes = 100

# Environment parameters
env_id = "FrozenLake-v1"
max_steps = 99
gamma = 0.95
eval_seed = []

# Exploration parameters
max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.0005

## Training tldr;

Create a loop for training episodes.

We will first reduce epsilon. As we need less and less exploration and more exploitation with every episode.

Reset the environment.

Create a nested loop for the maximum steps.

Choose the action using the epsilon greedy policy.

Take action (At) and observe the expected reward(Rt+1) and state(St+1).

Take the action (a) and observe the outcome state(s') and reward (r).

Update the Q-function using the formula.

If done= True, finish the episode and break the loop.

Finally, change the current state to a new state.

After completing all of the training episodes, the function will return the updated Q-Table.

In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in trange(n_training_episodes):

    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reset the environment
    state = env.reset()
    step = 0
    done = False

    # repeat
    for step in range(max_steps):

      action = epsilon_greedy_policy(Qtable, state, epsilon)


      new_state, reward, done, info = env.step(action)


      Qtable[state][action] = Qtable[state][action] + learning_rate * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action])

      # If done, finish the episode
      if done:
        break

      # Our state is the new state
      state = new_state
  return Qtable

In [ ]:
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

In [ ]:
Qtable_frozenlake

## Visualizing result as GIF

In [ ]:
def record_video(env, Qtable, out_directory, fps=1):
  images = []
  done = False
  state = env.reset(seed=random.randint(0,500))
  img = env.render(mode='rgb_array')
  images.append(img)
  while not done:
    # Take the action (index) that have the maximum expected future reward given that state
    action = np.argmax(Qtable[state][:])
    state, reward, done, info = env.step(action) # We directly put next_state = state for recording logic
    img = env.render(mode='rgb_array')
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
video_path="/content/replay.gif"
video_fps=1
record_video(env, Qtable_frozenlake, video_path, video_fps)

from IPython.display import Image
Image('./replay.gif')